In [ ]:
!pip install git+https://github.com/KarolKutyla/awp_tf_tests.git
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf

from awp_tf_tests.datasets.v2 import cifar_10

from awp_tf.attacks.v2.pgd import PGDAttack, PGDParams
from awp_tf import awp
from awp_tf import batch_processor
from awp_tf.callbacks import checkpoint_callback, epoch_logger

tf.config.run_functions_eagerly(False)

train_ds, test_ds = cifar_10.load_cifar_dataset()
steps_per_epoch = train_ds.cardinality()

attack_params = PGDParams(perturbation_bound=128/255, pgd_step=20, pgd_step_size=15/255, norm="l2")
pgd_20 = PGDAttack(attack_params)
protocol_params = batch_processor.AWPParams(alternate_iteration=1, awp_steps=1, weight_constraint=1.0e-2, step_size=1.0e-2)
awp_params = awp.Params(mode="trades", protocol_params=protocol_params)
params = awp.Params(protocol_params=protocol_params)

test_ds_attacked = test_ds.map(lambda x, y: (pgd_20.generate(x, y), y))

def get_checkpoint_path(model_name):
    return f"/content/drive/MyDrive/{model_name}/epoch_200.keras"

Visualizing adversarial examples on single batch from train dataset and evaluating trained model on test dataset

In [ ]:
clean_model = tf.keras.models.load_model(get_checkpoint_path("resnet_18v2_normal"))
clean_eval = clean_model.evaluate(test_ds_attacked)

Visualizing adversarial examples on single batch from train dataset and evaluating trained model on test dataset

In [ ]:
adv_model = tf.keras.models.load_model(get_checkpoint_path("resnet_18v2_adversarial"))

Visualizing adversarial examples on single batch from train dataset and evaluating trained model on test dataset

In [ ]:
awp_model = tf.keras.models.load_model(get_checkpoint_path("resnet_18v2_awp"))